# Combining Data

## Reading in the Data

Which manufacturer's planes made the most flights from NYC airports in November 2013?

To answer this question, we will need to combine information from two data sets.

The first is a data set of flights in November 2013...

In [ ]:
import pandas as pd
data_dir = "https://datasci112.stanford.edu/data/nycflights13/"
df_flights = pd.read_csv(f"{data_dir}/flights11.csv")
df_flights

...and the second is a data set of information about planes.

In [ ]:
df_planes = pd.read_csv(f"{data_dir}/planes.csv")
df_planes

## Joining Datasets on a Key

The Pandas function `.merge()` can be used to join two `DataFrame`s on a key.

In [ ]:
df_joined = df_flights.merge(df_planes, on="tailnum")
df_joined

By default, Pandas adds the suffixes **`_x`** and **`_y`** to overlapping column names, but this can be customized using the `suffixes=` option.

In [ ]:
df_joined = df_flights.merge(df_planes, on="tailnum",
                             suffixes=("_flight", "_plane"))
df_joined.columns

Now that we have joined the two data sets, we can answer the original question: which manufacturer's planes made the most flights from New York City airports in 2013?

In [ ]:
df_joined["manufacturer"].value_counts()

## Joining on Multiple Variables

What weather factors cause flight delays?

In [ ]:
df_weather = pd.read_csv(f"{data_dir}/weather.csv")
df_weather

Let’s start by looking at flights out of JFK. We need to join to the weather data on year, month, day, and hour.

In [ ]:
df_jfk = df_flights[df_flights["origin"] == "JFK"].merge(
    df_weather[df_weather["airport"] == "JFK"],
    on=("year", "month", "day", "hour"))
df_jfk

In [ ]:
df_jfk.plot.scatter(x="visib", y="dep_delay", alpha=0.2)

In [ ]:
df_jfk.groupby("visib")["dep_delay"].mean().plot.line()

Now let's do it for all the airports. The problem is that the airport is called **`origin`** in one data set and **`airport`** in the other.

In [ ]:
df_flights_weather = df_flights.merge(
    df_weather,
    left_on=("origin", "year", "month", "day", "hour"),
    right_on=("airport", "year", "month", "day", "hour"))
df_flights_weather

Notice that `df_flights_weather` contains columns called **`origin`** and **`airport`**.

Now that the data is joined, we can visualize the average departure delay as a function of the visibility at the three airports.

In [ ]:
(df_flights_weather
 .groupby(["airport", "visib"])["dep_delay"].mean()
 .unstack("airport")
 .plot.line())